In [51]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score, cross_val_predict
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, f1_score

from pathlib import Path
import warnings

In [52]:
warnings.filterwarnings("ignore")

In [53]:
def load_data(path: str) -> pd.DataFrame:
    data_path = Path(path)

    if not data_path.exists():
        raise FileNotFoundError(f'{data_path} does not exist')

    return pd.read_csv(data_path)

df = load_data("../data/processed-data/processed-data.csv")

In [54]:
TARGET = "is_mcu_canon"

df = df.drop(columns=["id", "age_years"])

x = df.drop(columns=[TARGET])
y = df[TARGET]

In [55]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [56]:
def evaluate(model, x, y):
    f1_score = cross_val_score(model, x, y, cv=kf, scoring="f1")
    acc_score = cross_val_score(model, x, y, cv=kf, scoring="accuracy")
    return  (
        round(f1_score.mean(), 3), round(f1_score.std(), 3),
        round(acc_score.mean(), 3), round(acc_score.std(), 3),
    )

models = {
    "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "RandomForestClassifier": RandomForestClassifier(max_depth=4, min_samples_leaf=5, class_weight="balanced", random_state=42),
    "XGBClassifier": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    f1_mean, f1_std, acc_mean, acc_std = evaluate(model, x, y)
    results.append([name, f1_mean, f1_std, acc_mean, acc_std])

results_df = pd.DataFrame(results, columns=["Model", "F1_mean", "F1_std", "Acc_mean", "Acc_std"])
print(results_df)

                    Model  F1_mean  F1_std  Acc_mean  Acc_std
0      LogisticRegression    0.772   0.051     0.746    0.048
1  RandomForestClassifier    0.769   0.052     0.740    0.043
2           XGBClassifier    0.812   0.053     0.789    0.062


In [57]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
}

grid_search = GridSearchCV(XGBClassifier(random_state=42), param_grid, cv=kf, scoring="f1", n_jobs=-1)
grid_search.fit(x, y)
print("==========XGB GRID SEARCH RESULTS==========")
print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_}\n")

fi = pd.Series(grid_search.best_estimator_.feature_importances_, index=x.columns).sort_values(ascending=False)
print("==========XGB FI==========")
print(fi.head())

==========XGB GRID SEARCH RESULTS==========
Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100}
Best score: 0.8329497079497079

==========XGB FI==========
year            0.167029
TMDB            0.124155
budget_log      0.111151
decade_2010s    0.107675
type_movie      0.060384
dtype: float32


In [58]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

final_model = XGBClassifier(learning_rate=0.05, max_depth=4, n_estimators=100, random_state=42)
final_model.fit(x_train, y_train)

y_pred = final_model.predict(x_test)
y_proba_cv = cross_val_predict(final_model, x, y, cv=kf, method="predict_proba")[:, 1]

print("==========FINAL MODEL PROBA REPORT==========")
for threshold in [0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred_cv = (y_proba_cv >= threshold).astype(int)
    print(f"Threshold {threshold}: F1 = {f1_score(y, y_pred_cv):.3f}")

print("\n==========FINAL MODEL REPORT==========")
print(classification_report(y_test, y_pred))


def predict_with_threshold(model, x, threshold=0.3):
    proba = model.predict_proba(x)[:, 1]
    return (proba >= threshold).astype(int)

y_pred_final = predict_with_threshold(final_model, x_test, threshold=0.3)

print("\n==========FINAL MODEL FINAL REPORT==========")
print(classification_report(y_test, y_pred_final))

==========FINAL MODEL PROBA REPORT==========
Threshold 0.25: F1 = 0.843
Threshold 0.3: F1 = 0.843
Threshold 0.35: F1 = 0.837
Threshold 0.4: F1 = 0.823
Threshold 0.45: F1 = 0.834
Threshold 0.5: F1 = 0.831

==========FINAL MODEL REPORT==========
              precision    recall  f1-score   support

           0       0.74      0.88      0.80        16
           1       0.86      0.71      0.77        17

    accuracy                           0.79        33
   macro avg       0.80      0.79      0.79        33
weighted avg       0.80      0.79      0.79        33


==========FINAL MODEL FINAL REPORT==========
              precision    recall  f1-score   support

           0       0.92      0.69      0.79        16
           1       0.76      0.94      0.84        17

    accuracy                           0.82        33
   macro avg       0.84      0.81      0.81        33
weighted avg       0.84      0.82      0.81        33

